In [8]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

url = 'https://apis.data.go.kr/1400000/forestStusService/getfirestatsservice'

params = {
    'serviceKey': 'Wyvg1VbEpufrNXq8BDq9rVWmLmu11p76xtaafVuiYYxk9nxduupS3U0Z4cbs4lG2FVVMK1SOlLPZVoe6pxgNIw==',
    'numOfRows': 100,
    'pageNo': 1,
    'searchStDt': '20160101',
    'searchEdDt': '20231231'
}

headers = {
    'User-Agent': 'Mozilla/5.0'
}

response = requests.get(url, params=params, headers=headers)
print(response.text)

<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg></header><body><items><item><damagearea>0.04</damagearea><endday>08</endday><endmonth>06</endmonth><endtime>17:00:00</endtime><endyear>2021</endyear><firecause>기타</firecause><locbunji>산9-2</locbunji><locdong>화상대</locdong><locgungu>홍천</locgungu><locmenu>내촌</locmenu><locsi>강원</locsi><startday>08</startday><startdayofweek>화요일</startdayofweek><startmonth>06</startmonth><starttime>14:52:00</starttime><startyear>2021</startyear></item><item><damagearea>0.01</damagearea><endday>07</endday><endmonth>06</endmonth><endtime>17:00:00</endtime><endyear>2021</endyear><firecause>쓰레기소각</firecause><locbunji>산126</locbunji><locdong>향호</locdong><locgungu>강릉</locgungu><locmenu>주문진</locmenu><locsi>강원</locsi><startday>07</startday><startdayofweek>월요일</startdayofweek><startmonth>06</startmonth><starttime>15:13:00</starttime><startyear>2021</startyear></item><item><damagear

In [9]:
import math
root = ET.fromstring(response.text)
total = int(root.find('.//totalCount').text)
total_pages = math.ceil(total / params['numOfRows'])
total_pages


46

In [13]:
import time
all_rows = []

for page in range(1, total_pages + 1):
    params['pageNo'] = page
    r = requests.get(url, params=params, headers=headers)
    root = ET.fromstring(r.text)

    for item in root.findall('.//item'):
        row = {child.tag: child.text for child in item}  # 모든 요소 포함
        all_rows.append(row)

    print(f"페이지 {page}/{total_pages} 완료")
    time.sleep(0.2)  # 서버 차단 방지

페이지 1/46 완료
페이지 2/46 완료
페이지 3/46 완료
페이지 4/46 완료
페이지 5/46 완료
페이지 6/46 완료
페이지 7/46 완료
페이지 8/46 완료
페이지 9/46 완료
페이지 10/46 완료
페이지 11/46 완료
페이지 12/46 완료
페이지 13/46 완료
페이지 14/46 완료
페이지 15/46 완료
페이지 16/46 완료
페이지 17/46 완료
페이지 18/46 완료
페이지 19/46 완료
페이지 20/46 완료
페이지 21/46 완료
페이지 22/46 완료
페이지 23/46 완료
페이지 24/46 완료
페이지 25/46 완료
페이지 26/46 완료
페이지 27/46 완료
페이지 28/46 완료
페이지 29/46 완료
페이지 30/46 완료
페이지 31/46 완료
페이지 32/46 완료
페이지 33/46 완료
페이지 34/46 완료
페이지 35/46 완료
페이지 36/46 완료
페이지 37/46 완료
페이지 38/46 완료
페이지 39/46 완료
페이지 40/46 완료
페이지 41/46 완료
페이지 42/46 완료
페이지 43/46 완료
페이지 44/46 완료
페이지 45/46 완료
페이지 46/46 완료


In [16]:
df = pd.DataFrame(all_rows)
df
num_cols = ['damagearea']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 날짜형
df['startdate'] = pd.to_datetime(df['startyear'] + '-' + df['startmonth'] + '-' + df['startday'], errors='coerce')
df['enddate'] = pd.to_datetime(df['endyear'] + '-' + df['endmonth'] + '-' + df['endday'], errors='coerce')



In [24]:
df.to_csv('산불_2016_2022데이터.csv',index=False)

In [25]:
df['startyear'].unique()

array(['2021', '2020', '2019', '2018', '2017', '2016', '2023', '2022'],
      dtype=object)